<a href="https://colab.research.google.com/github/io-uty/skt-pytorch/blob/main/Qwen3_VL_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import login
login()

In [ ]:
# ================================================================
# Qwen3-VL-8B 베이스 화장 불균일 탐지 — train/val 분리 + zero-shot/데이터만/증강 3조건 비교
# Colab 셀 단위로 나눠서 실행하세요 (### CELL 구분선 기준)
# ================================================================

### CELL 1: 설치
Colab 기본 tensorflow가 mediapipe(protobuf<5 필요)랑 protobuf 버전이 충돌해서 아예 지움
(이 파이프라인은 tensorflow 안 씀 - Unsloth는 PyTorch 기반)
wandb도 protobuf 5용으로 컴파일된 pb2를 갖고 있어서 protobuf가 4.25.9로 내려가면
`from wandb.proto.wandb_telemetry_pb2 import Imports` 에서 ImportError가 남 -> wandb도 같이 지움
(이 파이프라인은 wandb 로깅 안 씀, trainer의 report_to도 "none")
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-intel keras tensorboard wandb -q
!pip install unsloth albumentations mediapipe==0.10.14 -q
import os
os.kill(os.getpid(), 9)  # 설치 반영 위해 커널 강제 재시작 ("세션이 다운되었습니다" 메시지는 정상)
# 재시작되면 CELL 1 다시 실행하지 말고 바로 CELL 2부터 이어서 실행

### CELL 2: 공통 임포트 + 시스템 프롬프트
import json, os, re, random, unicodedata
import torch
from PIL import Image
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

random.seed(42)

SYSTEM_PROMPT = """당신은 사용자의 메이크업 베이스 화장 상태를 점검해주는 음성 어시스턴트입니다. 답변은 음성으로 전달되니 아래 규칙을 반드시 지키세요.

1. 점, 여드름, 모공, 잡티, 팔자주름, 심부볼(꺼진 볼), 홍조, 착색처럼 피부·얼굴 구조 자체의 특징은 절대 화장 문제로 판단하지 않습니다. 오직 파운데이션/베이스가 뭉치거나, 찍히거나, 덜 발려 경계가 생긴 경우만 판단합니다.
2. 위치는 좌표가 아니라 이마 위쪽/아래쪽/왼쪽/오른쪽, 미간, 코 왼쪽/오른쪽, 눈 아래 왼쪽/오른쪽, 입 위 왼쪽/오른쪽, 입 왼쪽/오른쪽, 턱처럼 큼직한 이목구비 기준 위/아래/왼쪽/오른쪽으로만 설명합니다. "광대"처럼 뼈인지 피부인지 헷갈리는 표현은 쓰지 않고, 손가락 마디 같은 세밀한 거리 단위도 쓰지 않습니다.
2-1. 문제 범위가 한 부위 전체(예: 볼 전체, 이마 전체)처럼 넓으면 세부 위치를 짚지 말고 "그 부위 전체적으로 고르게 펴 발라주세요"처럼 안내합니다.
3. 좌우는 반드시 사용자 본인 기준으로 말합니다. 셀카 이미지에서 화면 왼쪽에 보이는 얼굴은 사용자의 오른쪽 얼굴입니다. 좌우를 틀리면 엉뚱한 곳을 고치게 되니 항상 사용자 기준으로 변환해서 말합니다.
4. 결함이 여러 곳이면 심한 순서로 최대 두 곳까지만 안내하고, 나머지는 먼저 정리한 뒤 다시 찍어달라고 요청합니다.
5. 마크다운, 특수기호, 이모지, 번호 매기기, 좌표값은 절대 사용하지 않습니다. 한국어 구어체로 짧고 다정하게 문장을 끊어서 답합니다.
6. 결함이 없으면 고르게 잘 발렸다고 안심시켜 주고, 없는 문제를 만들어내지 않습니다.
7. 화장 상태 확인 외의 주제에는 답하지 않고, 화장을 도와주는 역할임을 부드럽게 안내합니다.
8. 얼굴이 화면에 다 안 나오거나 잘려 보이면, 휴대폰을 어느 방향으로 얼마나 움직이면 되는지 말로 안내합니다.
9. 사진이 너무 어두워서 판단하기 어려우면, 밝은 곳으로 이동하거나 불을 켠 뒤 다시 촬영해 달라고 요청합니다.
10. 판단이 확실하지 않으면 단정짓지 말고, 그 부분을 손끝으로 만졌을 때 두께감이나 결이 다르게 느껴지는지 되물어서 함께 확인합니다.
"""

QUESTION_TEMPLATES = [
    "이 얼굴 이미지에서 베이스 불균일(뭉침/찍힘/덜발림) 영역을 찾아줘",
    "베이스 어디가 뭉쳤어?",
    "지금 나 베이스 어때?",
    "화장 경계 티나는 부분 있어?",
    "피부 베이스 상태 체크해줘",
]

# --- 파일명 유니코드 정규화 문제 대응 ---
# merged_dataset.jsonl의 파일명은 NFC(완성형)인데, 맥에서 업로드된 실제 파일은 파일시스템이
# NFD(자모분리형)로 저장하는 경우가 있어서 눈에는 똑같이 보여도 os.path.exists가 실패함.
# -> 디렉토리 실제 파일 목록을 NFC로 정규화해서 매핑해두고 그걸로 찾음.
_dir_index_cache = {}

def _dir_index(image_dir, force_refresh=False):
    if force_refresh or image_dir not in _dir_index_cache:
        idx = {}
        if os.path.isdir(image_dir):
            for fname in os.listdir(image_dir):
                idx[unicodedata.normalize("NFC", fname)] = fname
        _dir_index_cache[image_dir] = idx
    return _dir_index_cache[image_dir]

def _find_in_dir(image_dir, filename):
    nfc = unicodedata.normalize("NFC", filename)
    idx = _dir_index(image_dir)
    if nfc not in idx:
        idx = _dir_index(image_dir, force_refresh=True)  # 증강 등으로 새로 생긴 파일 대응해서 한 번 더 스캔
    return idx.get(nfc)

def resolve_image_path(image_dir, filename):
    actual = _find_in_dir(image_dir, filename)
    return f"{image_dir}/{actual}" if actual is not None else f"{image_dir}/{filename}"

def image_exists(image_dir, filename):
    return _find_in_dir(image_dir, filename) is not None

# --- 얼굴 박스 검출 (사진마다 프레임 안에서 얼굴 차지 비율이 달라서, 사진 전체 기준 퍼센트로는
#     부위 이름이 자꾸 틀어짐 - 실제 감지된 얼굴 영역 기준으로 계산해야 정확함) ---
import mediapipe as mp
import cv2

_mp_face_detector = mp.solutions.face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)
_face_box_cache = {}

def get_face_box(image_path):
    if image_path in _face_box_cache:
        return _face_box_cache[image_path]
    img = cv2.imread(image_path)
    face_box = None
    if img is not None:
        h, w = img.shape[:2]
        result = _mp_face_detector.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if result.detections:
            box = result.detections[0].location_data.relative_bounding_box
            face_box = (box.xmin * w, box.ymin * h, (box.xmin + box.width) * w, (box.ymin + box.height) * h)
    _face_box_cache[image_path] = face_box
    return face_box

# 좌표 -> 부위 설명. face_box(감지된 얼굴 영역) 기준 상대 위치로 계산 - 없으면 이미지 전체 기준으로 fallback
# 주의: 좌표는 "화면에 보이는" 왼쪽/오른쪽이라 사용자 본인 기준과는 반대라서 여기서 뒤집어줌
# 명칭은 SYSTEM_PROMPT에 승인된 것만 사용: 이마 위/아래x좌/우, 미간, 코 좌/우, 눈 아래 좌/우, 입 위 좌/우, 입 좌/우, 턱
# "광대뼈"/"볼" 처럼 뼈-피부 구분이 헷갈리는 단어는 빼고, 눈/입 기준 위아래로 통일함
def describe_region(bbox, img_w, img_h, face_box=None):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    if face_box:
        fx1, fy1, fx2, fy2 = face_box
        fw, fh = max(fx2 - fx1, 1), max(fy2 - fy1, 1)
        cx_pct, cy_pct = (cx - fx1) / fw, (cy - fy1) / fh
    else:
        cx_pct, cy_pct = cx / img_w, cy / img_h  # fallback (얼굴 검출 실패시)
    h_pos = "오른쪽" if cx_pct < 0.4 else ("왼쪽" if cx_pct > 0.6 else "중앙")  # 이미지상 좌우 -> 사용자 기준 좌우로 반전

    if cy_pct < 0.15:
        return "이마 위쪽" if h_pos == "중앙" else f"이마 위쪽 {h_pos}"
    elif cy_pct < 0.32:
        return "미간" if h_pos == "중앙" else f"이마 아래쪽 {h_pos}"
    elif cy_pct < 0.50:
        return "코" if h_pos == "중앙" else f"코 {h_pos}"
    elif cy_pct < 0.59:
        return "눈 아래" if h_pos == "중앙" else f"눈 아래 {h_pos}"
    elif cy_pct < 0.68:
        return "입 위" if h_pos == "중앙" else f"입 위 {h_pos}"
    elif cy_pct < 0.85:
        return "입" if h_pos == "중앙" else f"입 {h_pos}"
    else:
        return "턱"

# bbox가 얼굴 너비/높이의 28% 이상을 차지하면 "국소 부위"가 아니라 "넓은 범위"로 판단
def is_wide_area(bbox, face_box):
    if not face_box:
        return False
    x1, y1, x2, y2 = bbox
    fx1, fy1, fx2, fy2 = face_box
    fw, fh = max(fx2 - fx1, 1), max(fy2 - fy1, 1)
    return (x2 - x1) / fw > 0.28 or (y2 - y1) / fh > 0.28

def make_answer(row, image_dir):
    if not row["boxes"]:
        return "전체적으로 베이스가 고르게 잘 발려 있어요. 지금 상태 그대로 두셔도 좋아요."
    img_path = resolve_image_path(image_dir, row["image"])
    try:
        w, h = Image.open(img_path).size
    except Exception:
        w, h = 3024, 4032  # fallback
    face_box = get_face_box(img_path)
    boxes = row["boxes"][:2]  # 최대 두 곳까지만
    regions = [(describe_region(b["bbox"], w, h, face_box), is_wide_area(b["bbox"], face_box)) for b in boxes]
    extra = len(row["boxes"]) - len(regions)

    parts = []
    for i, (r, wide) in enumerate(regions):
        if wide:
            desc = f"{r} 부위 전체적으로 베이스가 고르지 않게 발려 있어요."
        else:
            desc = f"{r} 쪽에 베이스가 고르지 않게 발려 있어요."
        parts.append(desc if i == 0 else "그리고 " + desc)
    if any(wide for _, wide in regions):
        parts.append("좁게 콕 집기보다 그 부위 전체를 손이나 퍼프로 넓게 펴 발라서 정리해 주세요.")
    else:
        parts.append("퍼프나 손끝으로 그 부분을 가볍게 두드려서 정리해 주세요.")
    parts.append("두드린 다음 손끝으로 쓸어봤을 때 주변과 두께 차이가 안 느껴지면 잘 된 거예요.")
    if extra > 0:
        parts.append("먼저 이 부분들부터 정리하시고, 다시 한번 찍어 보내주시면 나머지도 확인해 드릴게요.")
    return " ".join(parts)

# ================================================================
# 아직 미사용 - 준비만 해두는 코드. 학습/make_answer()에는 안 걸려있음.
# 나중에 부위별 라벨 데이터가 충분히 쌓이면 describe_region() 대신
# describe_region_fine()을 make_answer()/evaluate()에 연결하면 됨.
#
# FaceDetection(박스만)이 아니라 FaceMesh(468점 랜드마크)를 써서:
#  1) 콧대/코끝/인중/턱중앙처럼 세밀한 부위를 랜드마크 좌표 기준으로 판별하고
#  2) 좌우는 "이미지 가로 중심점" 대신, 코 중심선(랜드마크) 대비 위치로 판별해서
#     고개가 살짝 돌아간 사진에서도 더 안정적으로 사용자 기준 좌우를 맞춤
#
# 주의(확실하지 않음): 애교살/눈두덩이/눈 아래는 실제 얼굴에서도 서로 몇 mm 안 되는
# 거리라 랜드마크로도 확실히 구분되긴 어려움. 헤어라인도 FaceMesh가 "피부 표면"만
# 추적하고 머리카락 경계는 안 잡아주므로, 이마 위쪽 랜드마크에서 일정 비율 위로
# 추정하는 근사치일 뿐임. 이 두 그룹은 나중에 실제로 써보고 눈으로 검증 필요.
# ================================================================
_mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5)
_face_landmarks_cache = {}

def get_face_landmarks(image_path):
    if image_path in _face_landmarks_cache:
        return _face_landmarks_cache[image_path]
    img = cv2.imread(image_path)
    pts = None
    if img is not None:
        h, w = img.shape[:2]
        result = _mp_face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if result.multi_face_landmarks:
            lm = result.multi_face_landmarks[0].landmark
            pts = [(p.x * w, p.y * h) for p in lm]  # 픽셀 좌표로 변환
    _face_landmarks_cache[image_path] = pts
    return pts

def _anchor(pts, idxs):
    xs = [pts[i][0] for i in idxs]
    ys = [pts[i][1] for i in idxs]
    return sum(xs) / len(xs), sum(ys) / len(ys)

# 부위 이름 -> (센터용 랜드마크 묶음, "좌"쪽 랜드마크 묶음, "우"쪽 랜드마크 묶음)
# 센터만 있는 부위는 side 묶음 없이 anchor 하나만 사용
# "좌/우"는 실제 화면상 왼쪽/오른쪽이 아니라, 코 중심선 기준 어느 쪽인지로 판별 후
# SYSTEM_PROMPT 규칙 3(이미지 왼쪽 = 사용자 오른쪽)에 맞춰 마지막에 뒤집어줌
FINE_ZONE_ANCHORS = {
    "이마 중앙": {"center": [10]},
    "미간": {"center": [9]},
    "콧대": {"center": [6, 197, 195, 5, 168]},
    "코끝": {"center": [4, 1]},
    "인중": {"center": [2, 326, 97]},
    "턱 중앙": {"center": [152]},
    "이마": {"side_a": [103], "side_b": [332]},                       # 확실하지 않음(근사)
    "눈썹 위": {"side_a": [70], "side_b": [300]},
    "눈썹 바로 밑": {"side_a": [55], "side_b": [285]},
    "눈두덩이": {"side_a": [159, 158, 157], "side_b": [386, 385, 384]},
    "눈 아래": {"side_a": [145, 144], "side_b": [374, 373]},
    "애교살": {"side_a": [7, 163], "side_b": [249, 390]},             # 확실하지 않음(근사)
    "눈가 경계": {"side_a": [33], "side_b": [263]},
    "콧볼": {"side_a": [102, 49, 48, 115], "side_b": [331, 279, 278, 344]},
    "코 옆 얼굴경계": {"side_a": [205], "side_b": [425]},              # 확실하지 않음(근사)
    "광대 위": {"side_a": [116], "side_b": [345]},                    # 확실하지 않음(근사)
    "광대 아래": {"side_a": [192], "side_b": [416]},                  # 확실하지 않음(근사)
    "입술 옆 얼굴경계": {"side_a": [61], "side_b": [291]},
    "턱": {"side_a": [172, 136, 150, 149, 176, 148], "side_b": [397, 365, 379, 378, 400, 377]},
    "헤어라인 경계": {"center": [10]},  # 확실하지 않음 - 이마 위쪽 랜드마크에서 위로 추정해서 씀
}

def describe_region_fine(bbox, landmarks, nose_center_idx=(6, 168)):
    """landmarks: get_face_landmarks() 결과. 없으면 None 반환(호출부에서 describe_region()으로 폴백 필요)."""
    if not landmarks:
        return None
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    nose_x = sum(landmarks[i][0] for i in nose_center_idx) / len(nose_center_idx)

    best_name, best_dist, best_is_center = None, float("inf"), True
    for name, group in FINE_ZONE_ANCHORS.items():
        if "center" in group:
            ax, ay = _anchor(landmarks, group["center"])
            d = (cx - ax) ** 2 + (cy - ay) ** 2
            if d < best_dist:
                best_dist, best_name, best_is_center = d, name, True
        else:
            for side_key in ("side_a", "side_b"):
                ax, ay = _anchor(landmarks, group[side_key])
                d = (cx - ax) ** 2 + (cy - ay) ** 2
                if d < best_dist:
                    best_dist, best_name, best_is_center = d, name, False

    if best_is_center:
        return best_name
    # 코 중심선보다 이미지상 왼쪽에 있으면 -> 사용자 기준 오른쪽 (규칙 3과 동일한 반전)
    side_word = "오른쪽" if cx < nose_x else "왼쪽"
    return f"{best_name} {side_word}"

def to_conversation(row, image_dir="/content"):
    img_path = resolve_image_path(image_dir, row["image"])
    return {"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": random.choice(QUESTION_TEMPLATES)},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": make_answer(row, image_dir)}]},
    ]}

### CELL 3: train/val 분할 (결함 유무 기준 층화)
rows = [json.loads(l) for l in open("merged_dataset.jsonl", encoding="utf-8")]
pos = [r for r in rows if r["boxes"]]
neg = [r for r in rows if not r["boxes"]]
random.shuffle(pos); random.shuffle(neg)

def split(lst, ratio=0.85):
    k = int(len(lst) * ratio)
    return lst[:k], lst[k:]

train_pos, val_pos = split(pos)
train_neg, val_neg = split(neg)
train_rows, val_rows = train_pos + train_neg, val_pos + val_neg
random.shuffle(train_rows); random.shuffle(val_rows)

for name, rs in [("train.jsonl", train_rows), ("val.jsonl", val_rows)]:
    with open(name, "w", encoding="utf-8") as f:
        for r in rs: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"train {len(train_rows)} / val {len(val_rows)}")

### CELL 4: bbox 보존 이미지 증강 (train만, val은 증강 안 함)
import albumentations as A, cv2

AUG_MULTIPLIER = 4  # 원본 1장당 증강본 개수. 변형 종류가 4가지뿐이라 5~6 이상은 다양성 대비 효율이 떨어짐 (10배는 비추천)

transform_bbox = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.3),
    # A.Rotate 제거: 회전된 이미지에서 mediapipe 얼굴 검출이 부정확해져서
    # describe_region()이 계산하는 부위 라벨이 틀어질 수 있음 (region_acc 저하 원인으로 추정)
], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"]))

transform_plain = A.Compose([  # 정상 사진(박스 없음)용 - bbox 변환 불필요
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.3),
    # A.Rotate 제거 (위와 동일한 이유 - 일관성 위해 정상 사진도 회전 안 함)
])

def augment_row(row, image_dir, tag):
    img_path = resolve_image_path(image_dir, row["image"])
    img = cv2.imread(img_path)
    if img is None:
        return None
    new_name = f"aug_{tag}_{os.path.basename(row['image'])}"
    if row["boxes"]:
        bboxes = [b["bbox"] for b in row["boxes"]]
        labels = [b["label"] for b in row["boxes"]]
        result = transform_bbox(image=img, bboxes=bboxes, labels=labels)
        new_boxes = [{"label": l, "bbox": [round(c, 1) for c in b], "raw_label": row["boxes"][0].get("raw_label", "")}
                     for b, l in zip(result["bboxes"], result["labels"])]
    else:
        result = transform_plain(image=img)
        new_boxes = []
    cv2.imwrite(f"{image_dir}/{new_name}", result["image"])
    return {"image": new_name, "boxes": new_boxes}

aug_rows = []
total_attempts = len(train_rows) * AUG_MULTIPLIER
done = 0
for i, r in enumerate(train_rows):
    for k in range(AUG_MULTIPLIER):
        a = augment_row(r, "/content", f"{i}_{k}")
        if a:
            aug_rows.append(a)
        done += 1
        if done % 20 == 0 or done == total_attempts:
            print(f"진행중: {done}/{total_attempts} (성공 {len(aug_rows)}개)")

with open("train_augmented.jsonl", "w", encoding="utf-8") as f:
    for r in train_rows + aug_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"원본 {len(train_rows)} + 증강 {len(aug_rows)}({AUG_MULTIPLIER}배) = {len(train_rows) + len(aug_rows)}")

### CELL 5: 학습 함수 (데이터만 / 증강 포함 재사용)
def build_dataset(jsonl_path, image_dir="/content"):
    dataset, skipped = [], 0
    for l in open(jsonl_path, encoding="utf-8"):
        row = json.loads(l)
        if not image_exists(image_dir, row["image"]):
            skipped += 1; continue
        dataset.append(to_conversation(row, image_dir))
    print(f"{jsonl_path}: {len(dataset)}개 로드, {skipped}개 스킵")
    return dataset

from transformers import EarlyStoppingCallback

def train_model(train_jsonl, save_name, val_jsonl="val.jsonl", max_steps=200,
                 learning_rate=2e-4, r=16, lora_alpha=16, lora_dropout=0.05,
                 weight_decay=0.01, warmup_ratio=0.05, eval_steps=20, early_stopping_patience=4):
    model, tokenizer = FastVisionModel.from_pretrained(
        "unsloth/Qwen3-VL-8B-Instruct", load_in_4bit=True, use_gradient_checkpointing="unsloth")
    model = FastVisionModel.get_peft_model(
        model, finetune_vision_layers=True, finetune_language_layers=True,
        r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout, target_modules="all-linear")  # dropout으로 과적합 완화
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=build_dataset(train_jsonl),
        eval_dataset=build_dataset(val_jsonl),  # val loss로 과적합 실시간 확인
        args=SFTConfig(
            per_device_train_batch_size=2, gradient_accumulation_steps=4, max_steps=max_steps,
            learning_rate=learning_rate, weight_decay=weight_decay, warmup_ratio=warmup_ratio,
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            optim="adamw_8bit", output_dir=f"outputs_{save_name}",
            eval_strategy="steps", eval_steps=eval_steps,
            save_strategy="steps", save_steps=eval_steps, save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
            report_to="none",  # wandb 등 로깅 연동 끔 (wandb는 protobuf 충돌로 아예 삭제해서 안 씀)
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],  # val loss 안 좋아지면 자동 조기종료
    )
    trainer.train()  # 중간에 끊겨도 outputs_{save_name}/checkpoint-* 에 저장돼 있음
    model.save_pretrained(save_name)  # load_best_model_at_end=True라 val loss 제일 낮았던 시점 가중치로 저장됨
    tokenizer.save_pretrained(save_name)
    return model, tokenizer

model_data, tok_data = train_model("train.jsonl", "qwen3vl_data_only", max_steps=200)

# 증강으로 데이터가 커진 만큼 step 상한도 늘리되, early stopping이 있으니 실제로는 val loss 안 좋아지면 알아서 일찍 끝남
aug_size = sum(1 for _ in open("train_augmented.jsonl", encoding="utf-8"))
data_size = sum(1 for _ in open("train.jsonl", encoding="utf-8"))
aug_max_steps = min(int(200 * aug_size / data_size), 400)  # 상한 400으로 캡 - 너무 오래 걸리는 것 방지
print(f"증강 데이터 {aug_size}개 (원본의 {aug_size/data_size:.1f}배) -> max_steps 상한={aug_max_steps} (early stopping으로 더 일찍 끝날 수 있음)")

# 하이퍼파라미터는 이렇게 원하는 값으로 바꿔서 호출하면 됨 (예: 과적합 더 걱정되면 lora_dropout↑, r↓, learning_rate↓)
model_aug, tok_aug = train_model(
    "train_augmented.jsonl", "qwen3vl_augmented",
    max_steps=aug_max_steps, learning_rate=2e-4, r=16, lora_alpha=16,
    lora_dropout=0.05, weight_decay=0.01, eval_steps=20, early_stopping_patience=4,
)

### CELL 6: 평가 함수 (질문/답변 저장 + precision/recall/F1 + 부위 언급 정확도)
# 좌표를 답변에 안 넣기로 했으니 IoU 대신, 정답 bbox가 가리키는 큰 부위 이름이
# 모델 답변 텍스트에 실제로 언급됐는지로 "위치를 맞게 짚었는지"를 판단함 (근사치)
NO_DEFECT_PHRASES = ["고르게 잘 발려", "고르게 잘 발렸", "그대로 두셔도"]
DEFECT_KEYWORDS = ["고르지 않게", "뭉쳐", "찍힘", "덜 발", "경계", "정리해"]
REGION_HEADS = ["이마", "미간", "코", "눈", "입", "턱"]

def is_defect_mentioned(answer):
    if any(p in answer for p in NO_DEFECT_PHRASES):
        return False
    return any(k in answer for k in DEFECT_KEYWORDS)

def region_match(gt_region, answer):
    head = next((h for h in REGION_HEADS if h in gt_region), None)
    return head is not None and head in answer

def generate_answer(model, tokenizer, image_path, question, gen_kwargs=None):
    gen_kwargs = gen_kwargs or {}
    defaults = dict(max_new_tokens=256, temperature=0.3, top_p=0.9, top_k=50, repetition_penalty=1.1, do_sample=True)
    defaults.update(gen_kwargs)  # 넘긴 값으로 덮어쓰기

    FastVisionModel.for_inference(model)
    image = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]},
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    output = model.generate(**inputs, **defaults)
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True)  # 입력 프롬프트 부분은 잘라내고 새로 생성된 답변만

def evaluate(model, tokenizer, val_jsonl, name, image_dir="/content", gen_kwargs=None):
    questions = ["베이스 어디가 뭉쳤어?", "지금 나 베이스 어때?"]
    rows = [json.loads(l) for l in open(val_jsonl, encoding="utf-8")]
    results = []
    tp = fp = fn = tn = 0
    region_hits, region_total = 0, 0
    for r in rows:
        if not image_exists(image_dir, r["image"]):
            continue
        img_path = resolve_image_path(image_dir, r["image"])
        q = random.choice(questions)
        answer = generate_answer(model, tokenizer, img_path, q, gen_kwargs)

        gt_boxes = [b["bbox"] for b in r["boxes"]]
        has_gt, has_pred = bool(gt_boxes), is_defect_mentioned(answer)

        tp += has_gt and has_pred
        fn += has_gt and not has_pred
        fp += (not has_gt) and has_pred
        tn += (not has_gt) and not has_pred

        gt_regions = []
        if gt_boxes:
            try:
                w, h = Image.open(img_path).size
            except Exception:
                w, h = 3024, 4032
            face_box = get_face_box(img_path)
            gt_regions = [describe_region(b, w, h, face_box) for b in gt_boxes]
            for gr in gt_regions:
                region_total += 1
                if region_match(gr, answer):
                    region_hits += 1

        results.append({"image": r["image"], "question": q, "answer": answer,
                         "gt_regions": gt_regions})

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    region_acc = region_hits / region_total if region_total else 0
    metrics = {"name": name, "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
               "region_acc": round(region_acc, 3), "tp": tp, "fp": fp, "fn": fn, "tn": tn}

    json.dump({"metrics": metrics, "results": results}, open(f"eval_{name}.json", "w", encoding="utf-8"),
               ensure_ascii=False, indent=2)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    return metrics

### CELL 7: zero-shot / 데이터만 / 증강 3조건 비교 실행
all_metrics = []

base_model, base_tok = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct", load_in_4bit=True)
all_metrics.append(evaluate(base_model, base_tok, "val.jsonl", "zero_shot"))
all_metrics.append(evaluate(model_data, tok_data, "val.jsonl", "data_only"))
all_metrics.append(evaluate(model_aug, tok_aug, "val.jsonl", "augmented"))

import pandas as pd
df = pd.DataFrame(all_metrics)
print(df)
df.to_csv("comparison.csv", index=False)

train 313 / val 56
진행중: 20/1252 (성공 20개)
진행중: 40/1252 (성공 40개)
진행중: 60/1252 (성공 60개)
진행중: 80/1252 (성공 80개)
진행중: 100/1252 (성공 100개)
진행중: 120/1252 (성공 120개)
진행중: 140/1252 (성공 140개)
진행중: 160/1252 (성공 160개)
진행중: 180/1252 (성공 180개)
진행중: 200/1252 (성공 200개)
진행중: 220/1252 (성공 220개)
진행중: 240/1252 (성공 240개)
진행중: 260/1252 (성공 260개)
진행중: 280/1252 (성공 280개)
진행중: 300/1252 (성공 300개)
진행중: 320/1252 (성공 320개)
진행중: 340/1252 (성공 340개)
진행중: 360/1252 (성공 360개)
진행중: 380/1252 (성공 380개)
진행중: 400/1252 (성공 400개)
진행중: 420/1252 (성공 420개)
진행중: 440/1252 (성공 440개)
진행중: 460/1252 (성공 460개)
진행중: 480/1252 (성공 480개)
진행중: 500/1252 (성공 500개)
진행중: 520/1252 (성공 520개)
진행중: 540/1252 (성공 540개)
진행중: 560/1252 (성공 560개)
진행중: 580/1252 (성공 580개)
진행중: 600/1252 (성공 600개)
진행중: 620/1252 (성공 620개)
진행중: 640/1252 (성공 640개)
진행중: 660/1252 (성공 660개)
진행중: 680/1252 (성공 680개)
진행중: 700/1252 (성공 700개)
진행중: 720/1252 (성공 720개)
진행중: 740/1252 (성공 740개)
진행중: 760/1252 (성공 760개)
진행중: 780/1252 (성공 780개)
진행중: 800/1252 (성공 800개)
진행중: 820/1252 (성공 820개)
진행중: 

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Model does not have a default image size - using 512


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


...

Step,Training Loss,Validation Loss
20,1.215459,1.083232
40,0.018036,0.013720
60,0.008570,0.008657
80,0.007171,0.007600
100,0.008143,0.006536
120,0.005851,0.006389
140,0.006808,0.006216
160,0.007843,0.006116
180,0.004032,0.006433
200,0.006657,0.005890


Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-80/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-140/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpoint-160/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen3vl_augmented/checkpo

==((====))==  Unsloth 2026.8.18: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

{
  "name": "zero_shot",
  "precision": 0.947,
  "recall": 0.818,
  "f1": 0.878,
  "region_acc": 0.314,
  "tp": 36,
  "fp": 2,
  "fn": 8,
  "tn": 10
}
{
  "name": "data_only",
  "precision": 0.977,
  "recall": 0.977,
  "f1": 0.977,
  "region_acc": 0.657,
  "tp": 43,
  "fp": 1,
  "fn": 1,
  "tn": 11
}


In [ ]:

model_data.push_to_hub("iouty/qwen3vl-makeup-data-only", private=True)
tok_data.push_to_hub("iouty/qwen3vl-makeup-data-only", private=True)
model_aug.push_to_hub("iouty/qwen3vl-makeup-augmented", private=True)
tok_aug.push_to_hub("iouty/qwen3vl-makeup-augmented", private=True)